# 3.3 EDA — Riesgo de prórroga en contratos públicos (SECOP II)

**Base v1_A** (corte 26-oct-2026): 452.494 contratos firmados entre 2019 y 2025, muestra del 10 % por dígito 4 del `id_contrato`, autoponderada. Fuente: SECOP II, datos.gov.co (`jbjy-vk9h`, `p6dx-8zbt`), licencia CC BY-SA 4.0.

**Etiqueta v1:** `y` es la etiqueta de ampliación del plazo (tasa 27,6 %); `y_dias` y `y_modif` se usan solo para comparar. Cada sección que depende de `y` cierra con su interpretación (Decisión / Alternativas / Por qué / Evidencia). Las tablas numeradas que se citan en ellas están en `docs/eda_tablas.md`.

**Alcance:** §2 calidad, §3 distribuciones/deriva/correlación, §4 duración y madurez, §5 etiqueta, §6 H1–H7, §7 privacidad y tipo de documento, §8 fuga descriptiva y §9 decisiones. §4.3, §5–§6, §7.2 y §8 traen su interpretación después de la tabla o la gráfica.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import eda
from src import features as F
from src.base import validar_esquema

eda.config_graficas()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE = RAIZ / "data" / "processed" / "v1_A" / "base_contratos.parquet"
base, meta = eda.cargar_base(BASE)
validar_esquema(base)
X = F.construir_features(base)
PREFIJO = eda.prefijo_provisional(meta)

print("base:", base.shape, "| features:", X.shape)
print("version:", meta["version"], "| corte:", meta["fechas_de_corte"]["fecha_de_corte_de_la_censura"])
print("anios:", sorted(base["anio_firma"].unique().tolist()))
print("entidades:", base["nit_entidad"].nunique(), "| maduros:", round(100 * base["eval_madura"].mean(), 1), "%")

base: (452494, 50) | features: (452494, 39)
version: v1 | corte: 2026-10-26
anios: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
entidades: 3034 | maduros: 99.2 %


## 1. Carga y mapa de hipótesis

La base cumple el esquema 2a y `construir_features` devuelve las 39 features. El EDA separa lo que no depende de la etiqueta (calidad, distribución, deriva, duración, privacidad) de lo que sí (H1–H7 y etiqueta).

| Hipótesis | Señal al firmar | Feature(s) | Sección |
|---|---|---|---|
| H1. Procesos apresurados | Días entre publicación y firma | `dias_publicacion_a_firma` | 6.1 |
| H2. Poca competencia | Respuestas, oferentes, invitados | `p_respuestas_al_procedimiento`, `p_conteo_de_respuestas_a_ofertas`, `p_proveedores_invitados` | 6.2 |
| H3. Modalidad | Directa frente a competitivas | `modalidad_de_contratacion`, `justificacion_modalidad_de` | 6.3 |
| H4. Anualidad | Firma en el último trimestre | `mes_firma`, `firma_ultimo_trimestre`, `fin_planeado_cruza_anio` | 6.4 |
| H5. Plazos subestimados | Duración planeada | `p_dur_dias`, `p_unidad_de_duracion` | 6.5 |
| H6. Hábitos de la entidad | Tasa histórica de la entidad | `nit_entidad` (TargetEncoder) | 6.6 |
| H7. Tipo de objeto | UNSPSC y tipo de contrato | `unspsc_segmento`, `unspsc_familia`, `tipo_de_contrato`, `len_objeto` | 6.7 |

`y` no se usa en §2–§4; solo en §4.3, §5, §6 y §7.2, que quedan marcadas.

## 2. Calidad de datos

No depende de la etiqueta: se puede cerrar con v0. Se re-ejecuta con v1 para refrescar cifras.

### 2.1 Faltantes

**Qué se observa.** Los faltantes son bajos y vienen en bloques: 349 contratos (0,08 %) no tienen fila de Procesos (todas las `p_*` faltan), `fecha_de_inicio_del_contrato` falta en 4.059 (0,90 %), la duración planeada en 799 (0,18 %) y `unspsc_familia`/`unspsc_segmento` en 3.806 (0,84 %), todos sin UNSPSC. `y_modif` no falta (0 %) porque la v1 trae Modificaciones; es esperado.

**Decisión.** Imputar dentro del `Pipeline`, con la mediana/moda aprendida solo en train, y usar indicadores de bloque (`sin_proceso`, `sin_inicio`, `sin_duracion`). No borrar filas: los faltantes son pocos y borrarlas sesgaría la muestra.

In [2]:
cols_base = [c for c in base.columns if c not in ["id_contrato", "proceso_de_compra"]]
faltantes_base = eda.resumen_faltantes(base, cols_base)
faltantes_x = eda.resumen_faltantes(X, F.FEATURES)

print("Columnas de la base con faltantes:")
display(faltantes_base[faltantes_base["faltantes"] > 0])
print("Features con faltantes:")
display(faltantes_x[faltantes_x["faltantes"] > 0])
eda.graficar_faltantes(faltantes_x, titulo="Faltantes por feature (base v0)")
plt.show()

Columnas de la base con faltantes:


,columna,tipo,n,faltantes,pct_faltantes,n_unicos
0,N_mes,float64,452494,452494,100.00,0
1,fecha_de_inicio_del_contrato,datetime64[us],452494,4059,0.90,2803
2,eval_dur_planeada_dias,float32,452494,799,0.18,935
3,eval_fin_planeado,datetime64[us],452494,799,0.18,3976
4,p_fecha_de_publicacion_del,datetime64[us],452494,380,0.08,2708
5,p_conteo_de_respuestas_a_ofertas,float64,452494,349,0.08,43
6,p_duracion,float64,452494,349,0.08,847
7,p_numero_de_lotes,float64,452494,349,0.08,136
8,p_precio_base,float64,452494,349,0.08,146892
9,p_proveedores_con_invitacion,float64,452494,349,0.08,162


Features con faltantes:


,columna,tipo,n,faltantes,pct_faltantes,n_unicos
0,dias_firma_a_inicio,float64,452494,4059,0.90,634
1,unspsc_familia,object,452494,3806,0.84,376
2,unspsc_segmento,object,452494,3806,0.84,56
3,fin_planeado_cruza_anio,float64,452494,799,0.18,2
4,p_dur_dias,float64,452494,799,0.18,935
5,dias_publicacion_a_firma,float64,452494,380,0.08,569
6,log_precio_base,float64,452494,349,0.08,146885
7,p_conteo_de_respuestas_a_ofertas,float64,452494,349,0.08,43
8,p_numero_de_lotes,float64,452494,349,0.08,136
9,p_proveedores_con_invitacion,float64,452494,349,0.08,162


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1079778822.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.2 Valores imposibles

**Qué se observa.** No hay `id_contrato` duplicados, fechas basura, PII ni columnas de fuga. Los problemas están en duración y montos: 24.566 contratos (5,43 %) tienen duración planeada ≤ 0, 125 (0,03 %) pasan de 3.650 días y el máximo es 4.078,9 millones de días; 10 precios base son negativos, 3.497 son cero y el máximo es 2,5 billones. Además, 1.291 contratos (0,29 %) tienen fin planeado anterior a la firma: en todos el inicio es anterior a la firma y la duración no alcanza a cubrir ese retroceso (289 tienen duración ≤ 0).

**Decisión.** Tratar duración ≤ 0 como faltante y recortar a [1, 3.650] días; no eliminar contratos. `construir_features` ya recorta `dias_firma_a_inicio` y `dias_publicacion_a_firma`, pero no `p_dur_dias`. Los precios negativos quedan en 0 después de `clip(lower=0)` en `log_precio_base`, así que no se pueden separar de los ceros reales: el Pipeline crea `precio_no_positivo = log_precio_base == 0` y documenta la limitación.

In [3]:
imposibles = eda.revisar_valores_imposibles(base)
display(imposibles[imposibles["n"] > 0].reset_index(drop=True))
print("Los chequeos que no aparecen dieron 0.")

,chequeo,n,pct,detalle
0,fecha_de_inicio nula,4059,0.90,
1,inicio mas de 365 dias antes de la firma,256,0.06,
2,inicio mas de 365 dias despues de la firma,102,0.02,
3,p_fecha_de_publicacion nula,380,0.08,
4,p_duracion nula,349,0.08,
5,p_duracion <= 0,24586,5.43,
6,duracion planeada nula,799,0.18,
7,duracion planeada <= 0,24566,5.43,
8,duracion planeada > 3650 dias,125,0.03,
9,p_precio_base nulo,349,0.08,


Los chequeos que no aparecen dieron 0.


### 2.3 Atípicos

**Qué se observa.** Las colas largas están en duración, precio, pago adelantado y competencia. La duración planeada (`eval_dur_planeada_dias`) tiene 5,19 % en cero y 1,44 % fuera de 1,5 IQR, con máximo de 83,4 millones de días; `p_duracion` tiene 7,81 % fuera de 1,5 IQR; `p_precio_base` tiene 14,6 % fuera de 1,5 IQR y máximo de 3,4 billones; `p_proveedores_invitados` llega a 7.388 y `p_respuestas_al_procedimiento` a 254. `len_objeto` está truncado en 500 caracteres.

**Decisión.** No eliminar outliers: son pocos y pueden ser señal. Para kNN/SVM, `log1p` + `RobustScaler`; para HGB, dejar los valores crudos y usar el recorte suave de §2.2. Recortar `p_proveedores_invitados` en el percentil 99,9 solo para modelos de distancia.

In [4]:
base_atipicos = base.assign(len_objeto=X["len_objeto"])
cols_atipicos = ["p_duracion", "eval_dur_planeada_dias", "p_precio_base", "valor_de_pago_adelantado",
                 "p_proveedores_invitados", "p_respuestas_al_procedimiento", "p_numero_de_lotes", "len_objeto"]
atipicos = eda.resumen_atipicos(base_atipicos, cols_atipicos)
display(atipicos.round(3))
eda.graficar_histogramas(
    base_atipicos, cols_atipicos, ncols=4,
    escala_log=("p_precio_base", "valor_de_pago_adelantado", "p_proveedores_invitados",
                "p_respuestas_al_procedimiento", "p_numero_de_lotes"),
    titulo="Atípicos: distribuciones recortadas p1-p99",
)
plt.show()

,columna,n,n_cero,pct_cero,n_neg,min,p1,p50,p99,max,n_iqr_alto,pct_iqr_alto
0,p_precio_base,452145,3497,0.77,10,"-34,693,400,216.00","810,960.00","19,600,000.00","3,529,646,400.00","2,500,616,113,000.00",62166,13.75
1,p_respuestas_al_procedimiento,452145,408894,90.43,0,0.00,0.00,0.00,9.00,254.00,43251,9.57
2,p_numero_de_lotes,452145,414147,91.60,0,0.00,0.00,0.00,37.00,216.00,37998,8.40
3,p_duracion,452145,24586,5.44,0,0.00,0.00,15.00,350.00,"135,963,450.00",36371,8.04
4,p_proveedores_invitados,452145,442688,97.91,0,0.00,0.00,0.00,4.00,"7,229.00",9457,2.09
5,eval_dur_planeada_dias,451695,24566,5.44,0,0.00,0.00,120.00,420.00,"4,078,903,552.00",4427,0.98
6,valor_de_pago_adelantado,452494,451976,99.89,0,0.00,0.00,0.00,0.00,"25,178,078,596.00",518,0.11
7,len_objeto,452494,0,0.00,0,1.00,40.00,234.00,500.00,500.00,0,0.00


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\133183317.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.4 Cardinalidad de las categóricas

**Qué se observa.** `nit_entidad` (2.568), `ciudad` (664), `unspsc_familia` (373) y `unspsc_segmento` (56) concentran la cardinalidad alta. En el resto, las categorías raras (<1 %) son pocas: `justificacion` 31 categorías/4,1 % de filas, `departamento` 14/6,2 %, `sector` 7/3,0 %, `tipo_de_contrato` 17/2,4 %, `modalidad` 7/1,8 %, `tipodocproveedor` 7/0,9 %. `ciudad` tiene 648 categorías raras y 24,8 % de las filas; `nit_entidad` tiene 2.559 categorías con <1 % y 77,1 % de las filas, pero es la señal de H6 y va con TargetEncoder, no con one-hot.

**Decisión.** Para one-hot/kNN/SVM: agrupar categorías <1 % en `OTROS` y mantener `No Definido` como categoría o faltante explícito. Para HGB: categorías nativas con tope (200) o códigos ordinales; `nit_entidad` solo con `TargetEncoder` dentro del `Pipeline`.

In [5]:
card = eda.cardinalidad_categoricas(X, F.FEATURES_CAT)
display(card.round(2))
eda.graficar_cardinalidad(card, top=15, titulo="Cardinalidad de las categóricas (base v0)")
plt.show()

,columna,n_unicos,nulos,top,top_pct,n_categorias_lt_1pct,pct_filas_lt_1pct
0,nit_entidad,3034,0,899999034,5.62,3024,76.59
1,ciudad,735,0,No Definido,21.37,718,26.26
2,unspsc_familia,376,3806,8011,63.52,364,14.94
3,unspsc_segmento,56,3806,80,72.00,50,10.90
4,justificacion_modalidad_de,41,0,Servicios profesionales y apoyo a la gestión,73.36,36,5.76
5,departamento,34,0,Distrito Capital de Bogotá,33.50,13,5.18
6,sector,26,0,Servicio Público,24.91,8,2.73
7,tipo_de_contrato,24,0,Prestación de servicios,85.75,19,2.64
8,modalidad_de_contratacion,15,0,Contratación directa,76.75,10,2.46
9,tipodocproveedor,10,0,Cédula de Ciudadanía,84.25,8,0.53


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1369142577.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Distribución de features y deriva por año

No depende de la etiqueta. Se revisan las 18 numéricas y las 21 categóricas, y se comparan 2019–2025 para detectar deriva.

### 3.1 Distribución de las numéricas

**Qué se observa.** La mayoría son cero-infladas: `log_pago_adelantado` 99,8 % en cero, `p_proveedores_invitados` 96,4 %, `p_numero_de_lotes` 91,1 %, `p_respuestas_al_procedimiento` 79,3 %. `p_dur_dias` tiene mediana 120 días y p95 333; `log_precio_base` mediana 16,79 (unos 19,5 millones) y p99 22,30 (unos 4,9 mil millones). El calendario está repartido; `firma_ultimo_trimestre` es 25 %.

**Decisión.** Para kNN/SVM: usar tal cual `log_precio_base` y `log_pago_adelantado` (ya vienen en `log1p`), aplicar `log1p` a los conteos crudos y `RobustScaler` después; para HGB: valores crudos y flags de cero. No imputar los ceros de competencia: son "sin respuesta", no faltantes.

In [6]:
num_plot = [c for c in F.FEATURES_NUM if X[c].nunique() > 1]
display(eda.distribucion_numericas(X, F.FEATURES_NUM).round(2))
eda.graficar_histogramas(
    X, num_plot, ncols=4,
    escala_log=("p_dur_dias", "p_numero_de_lotes", "p_proveedores_invitados",
                "p_respuestas_al_procedimiento", "p_respuestas_externas",
                "p_conteo_de_respuestas_a_ofertas", "p_proveedores_unicos_con"),
    titulo="Distribución de las features numéricas (p1-p99)",
)
plt.show()

,columna,n,media,std,min,p1,p5,p50,p95,p99,max,pct_cero
0,log_pago_adelantado,452494,0.02,0.62,0.00,0.00,0.00,0.00,0.00,0.00,23.95,99.89
1,len_objeto,452494,248.78,106.88,1.00,40.00,99.00,234.00,466.00,500.00,500.00,0.00
2,dias_firma_a_inicio,448435,2.88,17.27,-365.00,-8.00,0.00,1.00,13.00,37.00,365.00,32.91
3,mes_firma,452494,5.55,3.68,1.00,1.00,1.00,5.00,12.00,12.00,12.00,0.00
4,dia_semana,452494,2.29,1.53,0.00,0.00,0.00,2.00,5.00,6.00,6.00,15.56
5,firma_ultimo_trimestre,452494,0.20,0.40,0.00,0.00,0.00,0.00,1.00,1.00,1.00,80.13
6,fin_planeado_cruza_anio,451695,0.24,0.43,0.00,0.00,0.00,0.00,1.00,1.00,1.00,76.26
7,log_precio_base,452145,16.89,2.16,0.00,13.61,14.91,16.79,20.22,21.98,28.55,0.78
8,p_dur_dias,451695,"12,066.52","6,298,973.53",0.00,0.00,0.00,120.00,337.00,420.00,"4,078,903,500.00",5.44
9,dias_publicacion_a_firma,452114,5.88,26.67,0.00,0.00,0.00,1.00,23.00,70.00,"1,000.00",36.63


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\4112744921.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.2 Composición de las categóricas

**Qué se observa.** El contrato típico es prestación de servicios (85,75 %), contratación directa (76,75 %), proveedor con cédula (84,25 %), en Bogotá/Valle/Antioquia. `No Definido` aparece en `ciudad` 21,37 %, `departamento` 1,79 %, `destino_gasto` 0,28 % y `tipodocproveedor` 0,16 %. `documentos_tipo` usa "No" y "No definido" como categorías distintas; `destino_gasto` usa "No Definido" y "No aplica".

**Decisión.** Normalizar texto (minúsculas, sin espacios) y unificar "No definido"/"No Definido" en una sola categoría; `No Definido` en `ciudad` se trata como categoría (no como faltante) porque puede informar la calidad del registro. Las categorías raras se agrupan como en §2.4.

In [7]:
cats_plot = ["tipo_de_contrato", "modalidad_de_contratacion", "justificacion_modalidad_de",
             "tipodocproveedor", "sector", "departamento", "ciudad", "p_unidad_de_duracion"]
eda.graficar_barras_categoricas(X, cats_plot, top=10, ncols=2,
                                titulo="Composición de las categóricas (top 10)")
plt.show()

C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\2099651677.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.3 Deriva numérica por año

**Qué se observa.** 2019 es atípico y 2020–2021 cambian la composición; 2023–2025 son más estables. Las medianas de duración se mueven entre 111 y 125 días; el precio base mediano baja de 17,33 (2019) a 16,64–16,80 (2020–2025); la competencia tiene ceros altos en todos los años (entre 39 % y 99 % según la variable). El cambio más fuerte está en las categóricas (§3.4), no en las numéricas.

**Decisión.** No usar `anio_firma` como feature. Ajustar imputación/escalado solo con train y validar temporalmente. En J3, probar como sensibilidad excluir 2019 del train; el plan fija 2019–2023, así que no se cambia hoy.

In [8]:
X_anio = X.assign(anio_firma=base["anio_firma"])
medianas, z, cambios = eda.deriva_numericas(X_anio, num_plot)
display(cambios.round(2))
eda.graficar_deriva_numericas(z, titulo="Deriva numérica: mediana estandarizada por año")
plt.show()

,mediana_inicio,mediana_fin,cambio_abs,z_max,cambio_2023_2024
dias_publicacion_a_firma,4.00,1.00,-3.00,2.45,0.00
log_precio_base,17.37,16.87,-0.50,2.36,-0.01
mes_firma,2.00,5.00,3.00,2.16,1.00
p_dur_dias,180.00,120.00,-60.00,2.06,-4.00
dias_firma_a_inicio,0.00,1.00,1.00,1.87,0.00
len_objeto,227.00,236.00,9.00,1.79,3.00
log_pago_adelantado,0.00,0.00,0.00,0.00,0.00
fin_planeado_cruza_anio,0.00,0.00,0.00,0.00,0.00
dia_semana,2.00,2.00,0.00,0.00,0.00
firma_ultimo_trimestre,0.00,0.00,0.00,0.00,0.00


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\3407481158.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.4 Deriva categórica por año

**Qué se observa.** Entre 2019 y 2025: cédula 44,6 %→82,4 %, NIT 53,1 %→17,1 %, contratación directa 24,2 %→72,5 %, prestación de servicios 52,0 %→82,9 %, UNSPSC 80 30,7 %→70,1 %, mínima cuantía 38,3 %→6,0 %. El salto ocurre en 2020–2021 y se estabiliza desde 2023.

**Decisión.** Misma que §3.3: no usar el año como feature; el preprocesamiento se ajusta con train. Reportar la deriva en §3.9 (monitoreo) y no comparar composiciones de 2019 con 2024 sin tenerla en cuenta.

In [9]:
cats_deriva = ["modalidad_de_contratacion", "tipo_de_contrato", "tipodocproveedor", "unspsc_segmento"]
display(eda.resumen_deriva_categoricas(X_anio, cats_deriva).round(2))
eda.graficar_evolucion_categorias(X_anio, cats_deriva, top=5, ncols=2,
                                  titulo="Evolución de la composición por año (top 5)")
plt.show()

,columna,top_inicio,pct_inicio,top_fin,pct_fin,cambio_max_pp,categoria_cambio
0,modalidad_de_contratacion,Contratación directa,59.57,Contratación directa,75.59,16.02,Contratación directa
1,unspsc_segmento,80,62.57,80,72.29,9.72,80
2,tipodocproveedor,Cédula de Ciudadanía,75.55,Cédula de Ciudadanía,84.85,9.29,Cédula de Ciudadanía
3,tipo_de_contrato,Prestación de servicios,79.59,Prestación de servicios,85.60,6.02,Prestación de servicios


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1066506475.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.5 Matriz de correlación de las numéricas

**Qué se observa.** Se excluye `p_proveedores_que_manifestaron` porque es constante 0. Con Spearman, hay dos pares casi duplicados: `p_proveedores_invitados` y `p_proveedores_con_invitacion` (r = 0,996; el segundo nunca es mayor y es igual en 99,59 % de las filas) y `p_respuestas_al_procedimiento` y `p_proveedores_unicos_con` (r = 0,995; iguales en 99,04 %). `mes_firma` y `firma_ultimo_trimestre` correlacionan 0,753 porque el flag es una transformación del mes. Con Pearson, `dias_firma_a_inicio` y `dias_publicacion_a_firma` dan −0,743, pero con Spearman solo −0,101: la correlación la producen los valores extremos, no el grueso de los datos. En Pearson, `p_respuestas_al_procedimiento` y `p_proveedores_unicos_con` siguen en 0,971, mientras que `p_proveedores_invitados` y `p_proveedores_con_invitacion` caen a 0,073 porque casi siempre valen 0 y su relación es de orden, no lineal.

**Qué implica para kNN y SVM.** La colinealidad no rompe el modelo, pero duplica el peso de la misma señal en la distancia y en el kernel. Para kNN/SVM: dejar un representante de cada par redundante (`p_proveedores_invitados` y `p_respuestas_al_procedimiento`), y para el calendario quedarse con `firma_ultimo_trimestre` o codificar el mes de forma cíclica. Para HGB, que parte el espacio por variable, la colinealidad importa menos: se pueden mantener las dos y la ablación de J3 decide.

**Decisión.** Para kNN/SVM: eliminar `p_proveedores_que_manifestaron` (constante 0), `p_proveedores_con_invitacion` y `p_proveedores_unicos_con`, y no usar `mes_firma` junto con `firma_ultimo_trimestre`. Para HGB: mantener todas y medir en J3. Evidencia: correlaciones y duplicación de filas.

In [10]:
num_corr = [c for c in F.FEATURES_NUM if X[c].nunique() > 1]
corr_s = eda.matriz_correlacion(X, num_corr, metodo="spearman")
pares_s = eda.pares_correlacionados(corr_s, umbral=0.7)
corr_p = X[num_corr].corr(method="pearson")
pares_p = eda.pares_correlacionados(corr_p, umbral=0.7)

print("Spearman |r| >= 0,7:")
with pd.option_context("display.float_format", "{:.3f}".format):
    display(pares_s)
print("Pearson |r| >= 0,7:")
with pd.option_context("display.float_format", "{:.3f}".format):
    display(pares_p)
eda.graficar_matriz_correlacion(corr_s, titulo="Correlación de Spearman entre features numéricas")
plt.show()

Spearman |r| >= 0,7:


,feature_1,feature_2,r,abs_r
0,p_proveedores_invitados,p_proveedores_con_invitacion,0.994,0.994
1,p_respuestas_al_procedimiento,p_proveedores_unicos_con,0.992,0.992


Pearson |r| >= 0,7:


,feature_1,feature_2,r,abs_r
0,p_respuestas_al_procedimiento,p_proveedores_unicos_con,0.983,0.983
1,mes_firma,firma_ultimo_trimestre,0.722,0.722


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\2974550288.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Duración planeada y madurez (censura)

La madurez es mecánica: `eval_madura = fin planeado + 60 días <= fecha de corte`, con `fin planeado = inicio (o firma) + duración del proceso`. No depende de la etiqueta; la relación entre madurez y `y` se ve en §4.3 con v1.

**Qué se observa.** La duración mediana es 116–180 días; el 5,43 % tiene duración ≤ 0 y el 0,18 % no tiene duración. La fracción madura va de 99,99 % (2019) a 97,64 % (2025); en total 99,2 %. Los 3.455 inmaduros son 11 sin duración, 867 con duración 0 y 2.577 con fin planeado + 60 días después del corte.

**Decisión.** Entrenar y evaluar solo con `eval_madura = True`; imputar `p_dur_dias` para las features, pero no usar `eval_*` como features. Reportar métricas por tramo de duración (≤ 3, 3–6, 6–12, > 12 meses) en la evaluación (requisito 3.7).

In [11]:
por_anio, por_tramo = eda.resumen_duracion_madurez(base)
display(por_anio)
display(por_tramo)
eda.graficar_duracion_madurez(base, por_anio, por_tramo)
plt.show()

,n,dur_mediana,dur_p95,pct_sin_duracion,pct_duracion_le0,pct_duracion_gt3650,pct_madura
anio_firma,,,,,,,
2019,14407,180.00,352.00,0.10,1.97,0.01,99.99
2020,36107,120.00,346.00,0.06,4.02,0.06,99.91
2021,56469,140.00,344.00,0.12,4.65,0.03,99.82
2022,71091,150.00,345.00,0.11,5.79,0.04,99.73
2023,83757,120.00,330.00,0.12,6.25,0.02,99.69
2024,88546,116.00,330.00,0.47,5.84,0.01,99.48
2025,102117,120.00,330.00,0.10,5.56,0.03,97.64


,n,pct,pct_madura
tramo,,,
<=30,42517,9.40,100.00
31-90,108000,23.87,100.00
91-180,144602,31.96,99.99
181-365,126781,28.02,99.35
366-3650,5104,1.13,68.53
>3650,125,0.03,0.00
Sin duracion / <=0,25365,5.61,96.54


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1830837622.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.3 Madurez y etiqueta

Se calcula la fracción madura por año y la tasa de `y` en maduros frente a no maduros, y por tramo de `eval_dur_planeada_dias` (duración 0 y sin dato separados). `eval_*` se usa solo para analizar; nunca entra como feature.


In [12]:
# §4.3: madurez y etiqueta. eval_* solo se usa para analizar; nunca como feature.
madurez_anio = (
    base.groupby("anio_firma")["eval_madura"]
    .agg(n="size", maduros="sum")
    .assign(pct_madura=lambda d: 100 * d["maduros"] / d["n"])
)
madurez_anio[["ic_inf_pct", "ic_sup_pct"]] = [
    [100 * x for x in eda.intervalo_wilson(r.maduros, r.n)]
    for r in madurez_anio.itertuples()
]
display(madurez_anio.round(2))

base_madurez = base.assign(
    madurez=pd.Categorical(
        base["eval_madura"].map({True: "Maduro", False: "No maduro"}),
        categories=["Maduro", "No maduro"],
        ordered=True,
    )
)
tabla_madurez_y = eda.tasas_en_pct(eda.tasa_por_grupo(base_madurez, "madurez")).round(2)
display(tabla_madurez_y)

dur = base["eval_dur_planeada_dias"]
tramo = pd.cut(
    dur.where(dur > 0),
    bins=[0, 30, 90, 180, 365, 3650, np.inf],
    labels=["1-30", "31-90", "91-180", "181-365", "366-3650", ">3650"],
    include_lowest=True,
)
tramo = tramo.cat.add_categories(["Duración 0", "Sin dato"])
tramo = tramo.mask(dur.eq(0), "Duración 0").fillna("Sin dato")
tramo = pd.Categorical(
    tramo,
    categories=["Duración 0", "1-30", "31-90", "91-180", "181-365", "366-3650", ">3650", "Sin dato"],
    ordered=True,
)
tabla_dur_y = eda.tasas_en_pct(
    eda.tasa_por_grupo(base.assign(tramo_dur=tramo), "tramo_dur")
).round(2)
display(tabla_dur_y.sort_values("tramo_dur").reset_index(drop=True))


,n,maduros,pct_madura,ic_inf_pct,ic_sup_pct
anio_firma,,,,,
2019,14407,14406,99.99,99.96,100.00
2020,36107,36076,99.91,99.88,99.94
2021,56469,56370,99.82,99.79,99.86
2022,71091,70899,99.73,99.69,99.77
2023,83757,83498,99.69,99.65,99.73
2024,88546,88088,99.48,99.43,99.53
2025,102117,99702,97.64,97.54,97.73


,madurez,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Maduro,449039,124272,27.68,27.54,27.81
1,No maduro,3455,776,22.46,21.10,23.88


,tramo_dur,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Duración 0,24566,6077,24.74,24.20,25.28
1,1-30,42517,8711,20.49,20.11,20.87
2,31-90,108000,24899,23.05,22.80,23.31
3,91-180,144602,40288,27.86,27.63,28.09
4,181-365,126781,42764,33.73,33.47,33.99
5,366-3650,5104,2155,42.22,40.87,43.58
6,>3650,125,34,27.20,20.17,35.59
7,Sin dato,799,120,15.02,12.71,17.66


**Decisión:** Usar `y` (ampliación del plazo; incluye suspensiones y ajustes de fecha, `docs/01_definicion_problema.md` §2) como etiqueta y filtrar `eval_madura = True` en entrenamiento y evaluación. El filtro está en `evaluacion.split_temporal`: ambos índices exigen `base["eval_madura"]`; el modelo se ajusta con esos índices y con `base["y"]` (`modelo.entrenar_v1`). Los no maduros no se usan como negativos ni se imputan, y `eval_madura` no entra como feature (no está en `FEATURES`, `features.FEATURES`). La etiqueta es `y = y_dias | y_modif` (`etiqueta.etiquetar`; metadatos en `modelo.metadatos`).

**Alternativas:**
- Incluir todos los contratos y usar `eval_madura` como feature o peso.
- Imputar `y = 0` a los no maduros.
- Modelo de supervivencia que use la censura (fuera de alcance): requeriría modelar el tiempo hasta la ampliación y los contratos que aún no la han tenido; el alcance v1 es clasificación binaria con contratos maduros (`docs/01_definicion_problema.md` §2).

**Por qué:** Un contrato no maduro no ha tenido tiempo de que se observe una ampliación; tratarlo como negativo confunde censura con ausencia de evento. La pérdida de muestra es pequeña: en train 2019–2023 solo 582 de 261.831 contratos (0,22 %) no son maduros (suma de `n no maduros` de la Tabla 4.3.4). La etiqueta aprobada ya incorpora las ampliaciones publicadas en Modificaciones (`etiqueta.etiquetar`).

**Evidencia:** Tabla 4.3.2: 3.455 no maduros (0,8 % del total de 2019–2025), con `y` 22,5 % frente a 27,7 % en maduros. Tabla 4.3.4: en 2019–2024 (n=1.040 no maduros), 12,4 % en no maduros frente a 27,8 % en maduros; en 2025 (n=2.415), 26,8 % frente a 27,1 %. Tabla 4.3.5: los no maduros de 2019–2024 se reparten entre comodato (305; `y` 1,6 %), Otro (398; `y` 7,0 %) y el resto de tipos (337; `y` 28,5 %).

**Cautela:** La comparación de 2025 no controla por duración; no se afirma que la censura no influya. La exclusión se sostiene por el diseño —tiempo de observación insuficiente—, no por esa comparación.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §4.3

## 5. Etiqueta `y` frente a `y_dias` por año

`y` es la etiqueta de ampliación del plazo; `y_dias` y `y_modif` se comparan por año. La tabla muestra las tasas y las discrepancias entre `y` y `y_dias`.

> **Nota.** `y`, `y_dias` y `y_modif` son etiquetas; no entran como features.


In [13]:
comp = eda.comparar_etiquetas(base, etiquetas=("y", "y_dias", "y_modif"))
display(eda.tasas_en_pct(comp).round(2))
display(eda.discrepancias_etiquetas(base))
print("Contratos con y_modif no nula:", int(base["y_modif"].notna().sum()))

,anio_firma,etiqueta,n,n_validos,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,2019,y,14407,14407,3192,22.16,21.49,22.84
1,2019,y_dias,14407,14407,1623,11.27,10.76,11.79
2,2019,y_modif,14407,14407,3188,22.13,21.46,22.81
3,2020,y,36107,36107,10592,29.34,28.87,29.81
4,2020,y_dias,36107,36107,4708,13.04,12.70,13.39
5,2020,y_modif,36107,36107,10581,29.30,28.84,29.78
6,2021,y,56469,56469,15698,27.80,27.43,28.17
7,2021,y_dias,56469,56469,5528,9.79,9.55,10.04
8,2021,y_modif,56469,56469,15693,27.79,27.42,28.16
9,2022,y,71091,71091,20907,29.41,29.07,29.74


,anio_firma,n,comparables,a_1_b_0,a_0_b_1,iguales
0,2019,14407,14407,1569,0,12838
1,2020,36107,36107,5884,0,30223
2,2021,56469,56469,10170,0,46299
3,2022,71091,71091,14194,0,56897
4,2023,83757,83757,14565,0,69192
5,2024,88546,88546,11136,0,77410
6,2025,102117,102117,7241,0,94876


Contratos con y_modif no nula: 452494


In [14]:
eda.graficar_comparacion_etiquetas(
    comp, titulo="Tasa por año: y frente a y_dias (base v1)"
)
plt.show()


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\479363893.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Decisión:** Usar `y` como objetivo (`etiqueta.etiquetar`; `modelo.metadatos`, `modelo.entrenar_v1`). `y_dias` y `y_modif` no entran como features: no están en `FEATURES` (`features.FEATURES`) y se conservan solo para análisis de sensibilidad. La etiqueta v1 es la unión `y = y_dias | y_modif`.

**Alternativas:**
- Usar `y_dias` sola.
- Usar `y_modif` sola.
- Entrenar dos modelos separados o un modelo multi-etiqueta.

**Por qué:** `y_dias` capta solo el 48,2 % de los positivos (Tabla 5.4): deja fuera 64.759 contratos con `y=1` que solo aparecen en Modificaciones. `y_modif` coincide con `y` salvo 150 contratos (Tabla 5.3), así que la etiqueta descansa casi entera en Modificaciones; usar solo `y_dias` perdería más de la mitad de los positivos, y usar solo `y_modif` perdería los 150 que solo tienen días adicionados. La unión conserva los dos mecanismos.

**Evidencia:** Tabla 5.1: `y` 27,6 %, `y_dias` 13,3 % y `y_modif` 27,6 %; diferencia `y − y_dias` +14,3 pp. Tabla 5.2: 64.759 contratos con `y=1, y_dias=0` y 0 con `y=0, y_dias=1`. Tabla 5.3: 150 contratos con `y=1, y_modif=0`; ninguno con `y=0, y_modif=1`. Tabla 5.4: dentro de `y=1`, 48,2 % tiene `y_dias=1` (60.289 de 125.048).

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §5

## 6. Hipótesis H1–H7

Tasa de `y` por grupo con intervalo de Wilson del 95 %. Cada subsección deja la tabla y la gráfica; la interpretación y la decisión van al final. La base v1_A es autoponderada; las tasas agrupadas describen la muestra.


In [15]:
Xy = X.assign(y=base["y"], anio_firma=base["anio_firma"])

### 6.1 H1. Procesos apresurados

Tasa de `y` por tramos de `dias_publicacion_a_firma` (días entre la publicación del proceso y la firma). `Sin dato` agrupa los 380 contratos sin fecha de publicación.

In [16]:
tabla_h1 = eda.tasa_por_tramos(
    Xy, "dias_publicacion_a_firma",
    bins=[-np.inf, 0, 7, 15, 30, 60, 120, np.inf],
    labels=["0", "1-7", "8-15", "16-30", "31-60", "61-120", ">120"],
)
display(eda.tasas_en_pct(tabla_h1).round(2))
eda.graficar_tasa_por_grupo(
    tabla_h1, col_grupo="dias_publicacion_a_firma",
    titulo="H1: tasa de y por días entre publicación y firma",
)
plt.show()

,dias_publicacion_a_firma,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,0,165613,42237,25.50,25.29,25.71
1,1-7,216256,61277,28.34,28.15,28.53
2,8-15,33810,9410,27.83,27.36,28.31
3,16-30,21228,6629,31.23,30.61,31.85
4,31-60,9738,3898,40.03,39.06,41.01
5,61-120,3259,1193,36.61,34.97,38.28
6,>120,2210,328,14.84,13.42,16.38
7,Sin dato,380,76,20.00,16.29,24.31


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\987398828.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: parcial** frente a H1 (`docs/01_definicion_problema.md` §3). La hipótesis espera que los procesos más apresurados (menos días entre publicación y firma) terminen más con el plazo ampliado. En train, el tramo de 0 días queda por debajo de la base (26,6 % frente a 28,8 %) y el máximo está en 31–60 días (40,1 %); la asociación existe, pero no es monótona ni en la dirección de “más apresurado, más ampliación”.

**Decisión:** `dias_publicacion_a_firma` entra como numérica (`features.FEATURES_NUM`); se calcula como `(firma - publicación).dt.days.clip(-30, 1000)` (`features.construir_features`). En la familia `arbol` del pipeline v1 (`modelo.FAMILIA`, `modelo.construir_pipeline`) las numéricas pasan crudas (`preprocesamiento.construir_preprocesador`) y el HGB maneja los NaN de forma nativa (`preprocesamiento.__doc__`). No hay imputación por mediana ni indicador `sin_publicacion` en v1; esos pasos solo existen en la familia `distancia` (`preprocesamiento.construir_preprocesador`), que no es el artefacto.

**Alternativas:**
- Imputar la mediana y agregar un indicador de faltante (el EDA lo llama `sin_publicacion`; en `distancia` se implementa como `dias_publicacion_a_firma_faltante`, `preprocesamiento.ImputarConIndicador`): descartada en v1 porque el HGB maneja NaN de forma nativa (`preprocesamiento.__doc__`) y el faltante es 0,08 % en 2019–2025 (380 de 452.494) y 11 contratos en train.
- Excluirla y dejar que el calendario capture el proceso.
- Usar solo tramos (0, 1–7, 8–15, …): el EDA lo recomendaba para modelos lineales o kNN por la relación no monótona; en v1 entra la numérica cruda.
- Familia `distancia` (kNN/SVM) con imputación, indicador y escalado: existe en `modelado.MODELOS` y `notebooks/05_modelado.ipynb`, pero no es el artefacto v1.

**Por qué:** La asociación no es lineal: se concentra en 31–120 días y cae en los extremos. El dato está disponible al firmar y el faltante es mínimo.

**Evidencia principal (train 2019–2023):** Tabla 6.1.1.b (n = 261.831; y = 28,8 %): 0 días 26,6 % (n=91.896; -2,3 pp), 31–60 días 40,1 % (n=6.772; +11,3 pp), 61–120 35,7 % (n=2.449; +6,9 pp), >120 13,9 % (n=1.753; -15,0 pp); `Sin dato` 11.

**Contraste 2019–2025:** Tabla 6.1.1: 0 días 25,5 %, 31–60 40,0 %, >120 14,8 %; `Sin dato` 380 (20,0 %). La dirección se mantiene: el máximo sigue en 31–60 días y el tramo 0 sigue por debajo de la base.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.1

### 6.2 H2. Poca competencia

Tasa de `y` por tramos de respuestas al procedimiento y de proveedores invitados. Los conteos son cero-inflados: la mayoría de los procesos no registra respuestas ni invitados.

In [17]:
tabla_h2_resp = eda.tasa_por_tramos(
    Xy, "p_respuestas_al_procedimiento",
    bins=[-np.inf, 0, 1, 3, 9, np.inf],
    labels=["0", "1", "2-3", "4-9", "10+"],
)
tabla_h2_inv = eda.tasa_por_tramos(
    Xy, "p_proveedores_invitados",
    bins=[-np.inf, 0, 4, 9, np.inf],
    labels=["0", "1-4", "5-9", "10+"],
)
display(eda.tasas_en_pct(tabla_h2_resp).round(2))
display(eda.tasas_en_pct(tabla_h2_inv).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
eda.graficar_tasa_por_grupo(
    tabla_h2_resp, col_grupo="p_respuestas_al_procedimiento",
    titulo="H2: respuestas al procedimiento", ax=axes[0],
)
eda.graficar_tasa_por_grupo(
    tabla_h2_inv, col_grupo="p_proveedores_invitados",
    titulo="H2: proveedores invitados", ax=axes[1],
)
plt.show()

,p_respuestas_al_procedimiento,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,0,408894,112539,27.52,27.39,27.66
1,1,18009,4530,25.15,24.53,25.79
2,2-3,11449,2924,25.54,24.75,26.35
3,4-9,9530,2900,30.43,29.51,31.36
4,10+,4263,2087,48.96,47.46,50.46
5,Sin dato,349,68,19.48,15.67,23.96


,p_proveedores_invitados,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,0,442688,121527,27.45,27.32,27.58
1,1-4,4996,1602,32.07,30.79,33.37
2,5-9,1079,442,40.96,38.07,43.93
3,10+,3382,1409,41.66,40.01,43.33
4,Sin dato,349,68,19.48,15.67,23.96


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\4149392421.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: contraria a la hipótesis** H2 (`docs/01_definicion_problema.md` §3). La hipótesis espera más ampliaciones cuando hay poca competencia; los datos muestran lo contrario en el número de respuestas: 0 respuestas queda en la base (28,9 %) y 10+ sube a 50,2 %, mientras que 1–3 respuestas queda por debajo (24,7 %).

**Decisión:** De las siete features de competencia marcadas `por verificar` (`features.FEATURES_POR_VERIFICAR`), seis entran como numéricas crudas (`p_respuestas_al_procedimiento` y `p_proveedores_invitados` son las principales, `features.FEATURES_POR_VERIFICAR`; cálculo en `features.construir_features`); `p_proveedores_que_manifestaron` se descarta por ser constante 0 en el train (`preprocesamiento.CONSTANTES`). En la familia `arbol` no se imputan los ceros, no se aplica `log1p` y no se dicotomizan (`preprocesamiento.construir_preprocesador`); el HGB maneja los NaN (`preprocesamiento.__doc__`). J3 mantuvo las ocho `FEATURES_POR_VERIFICAR` en la lista (las siete de competencia y `log_pago_adelantado`, `features.FEATURES_NUM`); la ablación de quitarlas no cambió el PR-AUC 0,584 (`resultados/j3_decisiones.md`).

**Alternativas:**
- Excluirlas por la alta proporción de ceros.
- Usar solo una de las dos.
- Dicotomizar en “cero” frente a “alguna”, o en “10+” frente al resto.
- `log1p` + `RobustScaler` para kNN/SVM: existe en la familia `distancia` (`preprocesamiento.construir_preprocesador`) y en los modelos de `modelado.MODELOS`, pero no en v1.

**Por qué:** La mayoría de los contratos no registra competencia, pero la cola se asocia con más ampliaciones; la relación no es lineal y los árboles pueden capturarla. La ablación de J3 muestra que el aporte no cambia el PR-AUC al quitarlas, así que se mantienen en `FEATURES` por ser información disponible al firmar y sin costo de imputación.

**Evidencia principal (train 2019–2023):** Tabla 6.2.1.b (n = 261.831; y = 28,8 %): 0 respuestas 28,9 % (n=235.700; +0,0 pp), 1 24,7 % (n=10.079; -4,1 pp), 2–3 24,7 % (n=7.220; -4,2 pp), 4–9 30,1 % (n=6.163; +1,2 pp), 10+ 50,2 % (n=2.663; +21,4 pp). Tabla 6.2.2.b: 0 invitados 28,7 % (n=255.976; -0,2 pp), 1–4 31,2 % (n=3.040; +2,3 pp), 5–9 39,9 % (n=722; +11,1 pp), 10+ 41,1 % (n=2.087; +12,3 pp).

**Contraste 2019–2025:** Tabla 6.2.1: 0 respuestas 27,5 %, 10+ 49,0 %; Tabla 6.2.2: 0 invitados 27,5 %, 10+ 41,7 %. La dirección se mantiene.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.2

### 6.3 H3. Modalidad de contratación

Tasa de `y` por modalidad. Se muestran las 12 modalidades más frecuentes; el resto queda en `OTROS`.

In [18]:
tabla_h3 = eda.tasa_top_categorias(Xy, "modalidad_de_contratacion", top=12)
display(eda.tasas_en_pct(tabla_h3).round(2))
eda.graficar_tasa_por_grupo(
    tabla_h3, col_grupo="modalidad_de_contratacion",
    titulo="H3: tasa de y por modalidad de contratación", horizontal=True,
)
plt.show()

,modalidad_de_contratacion,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Contratación directa,347300,87635,25.23,25.09,25.38
1,Contratación régimen especial,60145,24479,40.70,40.31,41.09
2,Mínima cuantía,24468,4407,18.01,17.53,18.50
3,Contratación Directa (con ofertas),4833,1533,31.72,30.42,33.05
4,Selección Abreviada de Menor Cuantía,4623,1920,41.53,40.12,42.96
5,Selección abreviada subasta inversa,4141,1512,36.51,35.06,37.99
6,Contratación régimen especial (con ofertas),3463,1250,36.10,34.51,37.71
7,Concurso de méritos abierto,1284,846,65.89,63.25,68.43
8,Licitación pública,1184,699,59.04,56.21,61.80
9,Licitación pública Obra Publica,810,642,79.26,76.33,81.91


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\505292876.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: confirmada** para H3 (`docs/01_definicion_problema.md` §3). La modalidad separa tasas muy distintas, desde 81,3 % en licitación pública de obra hasta 17,2 % en mínima cuantía; la hipótesis no fija una dirección, solo que la modalidad influye, y eso se observa.

**Decisión:** `modalidad_de_contratacion` y `justificacion_modalidad_de` entran como categóricas (`features.FEATURES_CAT`). En la familia `arbol` se codifican con `OrdinalEncoder(max_categories=200)`, que agrupa las categorías infrecuentes en un solo código (`preprocesamiento.construir_preprocesador`); no hay one-hot ni una dicotomía directa/competitiva construida a mano. Las categorías nativas se descartaron porque la app explica con SHAP TreeExplainer (`resultados/j3_decisiones.md`).

**Alternativas:**
- Excluirla y dejar solo tipo de contrato o UNSPSC.
- Dicotomizar en contratación directa frente a competitiva.
- One-hot completo sin agrupar.
- Categorías nativas de HGB: +0,003 de PR-AUC cv, dentro del ruido, pero la app no puede explicarlas (`resultados/j3_decisiones.md`).

**Por qué:** La modalidad separa tasas muy distintas, pero puede estar confundida con el tipo de objeto y con el peso de la contratación directa. La codificación ordinal con tope de 200 deja que el HGB parta el espacio y evita estimaciones inestables de categorías con pocos casos.

**Evidencia principal (train 2019–2023):** Tabla 6.3.1.b (n = 261.831; y = 28,8 %): licitación pública de obra 81,3 % (n=561), concurso de méritos abierto 67,5 % (n=869), licitación pública 59,7 % (n=749), régimen especial 47,0 % (n=30.663), selección abreviada de menor cuantía 40,2 % (n=2.951), contratación directa 26,1 % (n=203.256), mínima cuantía 17,2 % (n=15.280).

**Contraste 2019–2025:** Tabla 6.3.1: licitación pública de obra 79,3 %, concurso de méritos 65,9 %, régimen especial 40,7 %, selección abreviada de menor cuantía 41,5 %, contratación directa 25,2 %, mínima cuantía 18,0 %. La dirección general se mantiene; **cambia el orden** entre régimen especial (47,0 %) y selección abreviada (40,2 %) en train, que en 2019–2025 aparecen como 40,7 % y 41,5 %.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.3

### 6.4 H4. Anualidad presupuestal

Tasa de `y` por mes de firma y por el indicador `firma_ultimo_trimestre` (1 = octubre–diciembre).

In [19]:
tabla_h4_mes = eda.tasa_por_grupo(Xy, "mes_firma")
tabla_h4_mes["mes_firma"] = tabla_h4_mes["mes_firma"].astype(int)
tabla_h4_flag = eda.tasa_por_grupo(Xy, "firma_ultimo_trimestre")
tabla_h4_flag["firma_ultimo_trimestre"] = tabla_h4_flag["firma_ultimo_trimestre"].astype(int)
tabla_h4_flag_plot = tabla_h4_flag.assign(
    firma_ultimo_trimestre=tabla_h4_flag["firma_ultimo_trimestre"].map({0: "Resto", 1: "Último trimestre"})
)
display(eda.tasas_en_pct(tabla_h4_mes).round(2))
display(eda.tasas_en_pct(tabla_h4_flag).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
eda.graficar_tasa_por_grupo(
    tabla_h4_mes, col_grupo="mes_firma",
    titulo="H4: tasa de y por mes de firma", ax=axes[0], mostrar_n=False,
)
eda.graficar_tasa_por_grupo(
    tabla_h4_flag_plot, col_grupo="firma_ultimo_trimestre",
    titulo="H4: último trimestre frente al resto", ax=axes[1],
)
plt.show()

,mes_firma,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,1,82120,27817,33.87,33.55,34.20
1,2,63897,22069,34.54,34.17,34.91
2,3,35669,11456,32.12,31.63,32.60
3,4,24962,7020,28.12,27.57,28.68
4,5,26881,7850,29.20,28.66,29.75
5,6,26565,7860,29.59,29.04,30.14
6,7,33062,8323,25.17,24.71,25.64
7,8,34231,8433,24.64,24.18,25.09
8,9,35200,6839,19.43,19.02,19.85
9,10,33551,6084,18.13,17.72,18.55


,firma_ultimo_trimestre,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,0,362587,107667,29.69,29.55,29.84
1,1,89907,17381,19.33,19.08,19.59


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\2231585569.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: contraria a la hipótesis** H4 (`docs/01_definicion_problema.md` §3). La hipótesis espera más ampliaciones en el último trimestre; la tabla muestra lo contrario: 18,7 % en el último trimestre frente a 31,2 % en el resto en train (19,3 % frente a 29,7 % en 2019–2025).

**Decisión:** `mes_firma`, `firma_ultimo_trimestre`, `fin_planeado_cruza_anio` y `dias_firma_a_inicio` entran como numéricas (`features.FEATURES_NUM`). `mes_firma` se calcula como `firma.dt.month`, entero 1–12 (`features.construir_features`), y en la familia `arbol` pasa crudo (`preprocesamiento.construir_preprocesador`); no se trata como cíclica ni como one-hot. La versión cíclica (seno/coseno) existe solo en la familia `distancia` (`preprocesamiento.CICLICAS`, `preprocesamiento.construir_preprocesador`). `anio_firma` no está en `FEATURES` (`features.FEATURES`).

**Alternativas:**
- Excluir el calendario.
- Usar solo el último trimestre.
- Incluir el año de firma (descartado por deriva).
- `mes_firma` cíclica u one-hot, como recomendaba el EDA: descartada en v1; la versión cíclica solo se usa en `distancia` (`preprocesamiento.CICLICAS`, `preprocesamiento.construir_preprocesador`) y el HGB puede aislar cualquier mes con cortes sobre el entero.

**Por qué:** El patrón estacional es marcado y está disponible al firmar. La hipótesis sobre el último trimestre no se cumple en la dirección esperada, pero el mes sigue separando tasas y entra al modelo. No se incluye el año para no introducir deriva.

**Evidencia principal (train 2019–2023):** Tabla 6.4.2.b (n = 261.831; y = 28,8 %): último trimestre 18,7 % (n=50.186; -10,1 pp) y resto 31,2 % (n=211.645; +2,4 pp). Tabla 6.4.1.b: máximo en febrero 36,4 % (n=36.205), mínimo en octubre 16,1 % (n=16.814); enero 33,8 %, noviembre 16,2 %.

**Contraste 2019–2025:** Tabla 6.4.2: último trimestre 19,3 % frente a resto 29,7 %; Tabla 6.4.1: febrero 34,5 %, enero 33,9 %, noviembre 16,4 %. La dirección se mantiene y se acentúa.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.4

### 6.5 H5. Plazos subestimados

Tasa de `y` por la duración planeada relativa a la mediana de su familia UNSPSC (`p_dur_dias / mediana de la familia`) y por la duración absoluta en días. La mediana de referencia se calcula con 2019–2023 y duración > 0.

In [20]:
dur_h5 = eda.duracion_relativa_objeto(Xy, anio_referencia=2023)
Xy = Xy.assign(duracion_relativa=dur_h5["duracion_relativa"])
tabla_h5_rel = eda.tasa_por_tramos(
    Xy, "duracion_relativa",
    bins=[-np.inf, 0, 0.5, 1, 1.5, 2, 3, np.inf],
    labels=["<=0", "(0, 0.5]", "(0.5, 1]", "(1, 1.5]", "(1.5, 2]", "(2, 3]", ">3"],
)
tabla_h5_abs = eda.tasa_por_tramos(
    Xy, "p_dur_dias",
    bins=[-np.inf, 0, 30, 90, 180, 365, 730, np.inf],
    labels=["<=0", "1-30", "31-90", "91-180", "181-365", "366-730", ">730"],
)
display(eda.tasas_en_pct(tabla_h5_rel).round(2))
display(eda.tasas_en_pct(tabla_h5_abs).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
eda.graficar_tasa_por_grupo(
    tabla_h5_rel, col_grupo="duracion_relativa",
    titulo="H5: duración relativa a la mediana de la familia", ax=axes[0],
)
eda.graficar_tasa_por_grupo(
    tabla_h5_abs, col_grupo="p_dur_dias",
    titulo="H5: duración planeada absoluta (días)", ax=axes[1],
)
plt.show()

,duracion_relativa,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,<=0,24099,5908,24.52,23.98,25.06
1,"(0, 0.5]",94724,18872,19.92,19.67,20.18
2,"(0.5, 1]",132540,36113,27.25,27.01,27.49
3,"(1, 1.5]",84427,25470,30.17,29.86,30.48
4,"(1.5, 2]",55873,22617,40.48,40.07,40.89
5,"(2, 3]",47255,12410,26.26,25.87,26.66
6,>3,8951,2200,24.58,23.70,25.48
7,Sin dato,4625,1458,31.52,30.20,32.88


,p_dur_dias,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,<=0,24566,6077,24.74,24.20,25.28
1,1-30,42517,8711,20.49,20.11,20.87
2,31-90,108000,24899,23.05,22.80,23.31
3,91-180,144602,40288,27.86,27.63,28.09
4,181-365,126781,42764,33.73,33.47,33.99
5,366-730,2951,1685,57.10,55.31,58.87
6,>730,2278,504,22.12,20.47,23.88
7,Sin dato,799,120,15.02,12.71,17.66


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\4132503948.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: contraria a la hipótesis** H5 (`docs/01_definicion_problema.md` §3). La hipótesis espera que los plazos más subestimados se amplíen más; la tabla muestra lo contrario: (0, 0,5] de la duración típica tiene 20,5 % y (1,5, 2] tiene 41,1 % en train (19,9 % frente a 40,5 % en 2019–2025).

**Decisión:** `p_dur_dias` entra como numérica (`features.FEATURES_NUM`, `features.construir_features`) y pasa por `DuracionLimpia` dentro del Pipeline (`preprocesamiento.DuracionLimpia`, `preprocesamiento.construir_preprocesador`): los valores 0 o negativos se vuelven NaN y generan el indicador `p_dur_dias_cero`, y el resto se recorta al cuantil 99,9 % aprendido del train (`preprocesamiento.CUANTIL_DURACION`, `preprocesamiento.DuracionLimpia`); en el ajuste v1 (train maduro 2019–2023) ese tope es 1.259 días. `p_unidad_de_duracion` entra como categórica (`features.FEATURES_CAT`) y se codifica con el mismo `OrdinalEncoder`. No hay recorte fijo a 3.650 días: 3.650 es solo un corte descriptivo de la Tabla 4.3.3. La duración relativa a la mediana de la familia no está en `FEATURES`; solo existe para el EDA (`eda.duracion_relativa_objeto`).

**Alternativas:**
- Excluir la duración y usar solo el calendario.
- Usar tramos en vez de la numérica.
- Incluir solo la duración relativa: queda como alternativa opcional, fuera del pipeline v1; no está en `FEATURES` y solo se calcula en `eda.duracion_relativa_objeto`.
- Recorte fijo a 3.650 días, como recomendaba el EDA: descartado; el Pipeline usa el p99,9 aprendido del train (`preprocesamiento.DuracionLimpia`, `preprocesamiento.CUANTIL_DURACION`), que se adapta a la distribución y no fija un corte arbitrario.
- `log1p` + `RobustScaler` para kNN/SVM: solo en la familia `distancia` (`preprocesamiento.construir_preprocesador`), no en v1.

**Por qué:** La relación no es monótona: sube hasta 366–730 días y cae en >730; la relativa muestra un máximo en (1,5, 2]. La duración 0 no es un plazo real y se trata como faltante, no como cero, con su indicador.

**Evidencia principal (train 2019–2023):** Tabla 6.5.1.b (n = 261.831): (0, 0,5] 20,5 % (n=51.465; -8,4 pp), (0,5, 1] 28,4 %, (1, 1,5] 31,6 %, (1,5, 2] 41,1 % (n=33.771; +12,2 pp), (2, 3] 27,8 %, >3 27,7 %, <=0 25,2 %. Tabla 6.5.2.b: <=0 25,3 % (n=13.712), 1–30 22,8 %, 31–90 23,1 %, 91–180 28,9 %, 181–365 34,7 %, 366–730 59,6 % (n=2.222), >730 28,5 % (n=1.381), sin dato 11,3 %.

**Contraste 2019–2025:** Tabla 6.5.1: (0, 0,5] 19,9 %, (1,5, 2] 40,5 %; Tabla 6.5.2: 366–730 57,1 %, >730 22,1 %, <=0 24,7 %. La dirección se mantiene.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.5

### 6.6 H6. Hábitos de la entidad

Tasa de `y` en 2024 por grupo de la tasa histórica de la entidad (2019–2023): `Sin historia`, `Tasa 0` y terciles `T1`–`T3` de las entidades con tasa > 0. `T1` es el tercil de menor tasa histórica y `T3` el de mayor; `Sin historia` agrupa las entidades que no aparecen antes de 2024. Los terciles se arman sobre el rank de la tasa para que los empates no los colapsen. La tabla principal agrega mínimo, mediana y máximo de la tasa histórica por grupo; el resumen por entidad muestra cómo los empates se reparten entre grupos.

In [21]:
tabla_h6, hist_h6 = eda.tasa_entidad_historica(base, anio_objetivo=2024)
resumen_h6 = (
    hist_h6.groupby("grupo", observed=False)
    .agg(entidades=("tasa", "size"), tasa_min=("tasa", "min"),
         tasa_mediana=("tasa", "median"), tasa_max=("tasa", "max"))
    .reset_index()
)
display(eda.tasas_en_pct(tabla_h6).round(2))
display(eda.tasas_en_pct(resumen_h6).round(2))
eda.graficar_tasa_por_grupo(
    tabla_h6, col_grupo="grupo_hist",
    titulo="H6: tasa 2024 por grupos de tasa histórica de la entidad",
)
plt.show()

,grupo_hist,n,positivos,n_entidades,tasa_pct,ic_inf_pct,ic_sup_pct,tasa_hist_min_pct,tasa_hist_mediana_pct,tasa_hist_max_pct
0,Sin historia,1833,211,317,11.51,10.13,13.05,NaN,NaN,NaN
1,Tasa 0,6460,735,553,11.38,10.63,12.18,0.00,0.00,0.00
2,T1,21231,3030,416,14.27,13.81,14.75,0.40,10.34,16.42
3,T2,31741,6513,423,20.52,20.08,20.97,16.48,22.56,30.59
4,T3,27281,11342,377,41.57,40.99,42.16,30.71,42.86,100.00


,grupo,entidades,tasa_min_pct,tasa_mediana_pct,tasa_max_pct
0,T1,439,0.40,10.34,16.42
1,T2,439,16.48,22.56,30.59
2,T3,439,30.71,42.86,100.00
3,Tasa 0,1017,0.00,0.00,0.00


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\3080191459.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: confirmada** para H6 (`docs/01_definicion_problema.md` §3). La tasa histórica de la entidad separa la tasa futura: T3 41,6 %, T2 20,5 %, T1 14,3 %, tasa 0 11,4 %, sin historia 11,5 % en test 2024.

**Decisión:** `nit_entidad` entra como categórica (`features.FEATURES_CAT`) y se codifica con `TargetEncoder(target_type="binary", smooth="auto", cv=StratifiedKFold(5, shuffle=True, random_state=config.SEED))` dentro del Pipeline (`preprocesamiento._target_encoder`, `preprocesamiento.construir_preprocesador`); las entidades no vistas reciben la media global del train (`preprocesamiento._target_encoder`). Los proxies de entidad (`sector`, `orden`, `rama`, `entidad_centralizada`, `departamento`, `ciudad`) entran como categóricas (`features.FEATURES_CAT`) y, como el resto de categóricas, usan `OrdinalEncoder(max_categories=200)` (`preprocesamiento.construir_preprocesador`). No se usa one-hot de entidad ni el nombre.

**Alternativas:**
- One-hot de entidad: inviable por cardinalidad (2.086 entidades en test 2024, Tabla 6.6.1).
- Excluir la entidad.
- Usar la tasa histórica calculada fuera del Pipeline: filtraría información y no se reajustaría con cada fold.
- Codificar `nit_entidad` con `OrdinalEncoder` como el resto: convertiría un código arbitrario en una escala ordinal; el `TargetEncoder` captura la tasa histórica, que es la señal de H6.

**Por qué:** La tasa histórica de la entidad se asocia con la tasa futura, y las entidades sin historia se separan con una tasa baja; el codificador debe aprenderla solo con train para no filtrar 2024. La cardinalidad alta hace inviable el one-hot. El `cv` interno evita fuga dentro del train.

**Evidencia:** Tabla 6.6.1 (test 2024; historia 2019–2023; n=88.546): T3 41,6 %, T2 20,5 %, T1 14,3 %, tasa 0 11,4 %, sin historia 11,5 %. Tabla 6.6.2: 439 entidades en cada tercil y 1.017 en tasa 0. Esta sección usa test 2024, no las tablas `.b` de train.

**Cautela:** La evidencia sale de test 2024, no de train; la tasa histórica se calcula solo con 2019–2023. No se usa 2024 para ajustar el codificador.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.6

### 6.7 H7. Tipo de objeto

Tasa de `y` por segmento y familia UNSPSC, por tipo de contrato y por longitud del objeto (proxy de complejidad). En segmento, familia y tipo se muestran las 10 categorías más frecuentes; el resto queda en `OTROS`.

In [22]:
tabla_h7_seg = eda.tasa_top_categorias(Xy, "unspsc_segmento", top=10)
tabla_h7_fam = eda.tasa_top_categorias(Xy, "unspsc_familia", top=10)
tabla_h7_tipo = eda.tasa_top_categorias(Xy, "tipo_de_contrato", top=10)
tabla_h7_len = eda.tasa_por_tramos(
    Xy, "len_objeto",
    bins=[-np.inf, 50, 100, 200, 400, np.inf],
    labels=["<=50", "51-100", "101-200", "201-400", ">400"],
)
display(eda.tasas_en_pct(tabla_h7_seg).round(2))
display(eda.tasas_en_pct(tabla_h7_fam).round(2))
display(eda.tasas_en_pct(tabla_h7_tipo).round(2))
display(eda.tasas_en_pct(tabla_h7_len).round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
eda.graficar_tasa_por_grupo(
    tabla_h7_seg, col_grupo="unspsc_segmento",
    titulo="H7: segmento UNSPSC", horizontal=True, ax=axes[0],
)
eda.graficar_tasa_por_grupo(
    tabla_h7_fam, col_grupo="unspsc_familia",
    titulo="H7: familia UNSPSC", horizontal=True, ax=axes[1],
)
eda.graficar_tasa_por_grupo(
    tabla_h7_tipo, col_grupo="tipo_de_contrato",
    titulo="H7: tipo de contrato", horizontal=True, ax=axes[2],
)
plt.show()

,unspsc_segmento,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,80,323045,87160,26.98,26.83,27.13
1,85,26352,10147,38.51,37.92,39.09
2,93,21784,6951,31.91,31.29,32.53
3,81,11535,3249,28.17,27.35,28.99
4,86,9692,2247,23.18,22.35,24.03
5,72,7355,2858,38.86,37.75,39.98
6,84,3811,965,25.32,23.97,26.73
7,Sin dato,3806,1331,34.97,33.47,36.50
8,77,3605,821,22.77,21.43,24.17
9,78,3344,848,25.36,23.91,26.86


,unspsc_familia,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,8011,285020,77576,27.22,27.05,27.38
1,8510,19289,8429,43.70,43.00,44.40
2,8016,14181,3469,24.46,23.76,25.18
3,8010,12386,3279,26.47,25.70,27.26
4,9314,11844,3901,32.94,32.10,33.79
5,9315,8337,2680,32.15,31.15,33.16
6,8610,6485,1565,24.13,23.11,25.19
7,8012,5257,1178,22.41,21.30,23.56
8,8512,5081,1271,25.01,23.84,26.22
9,8110,4715,1833,38.88,37.49,40.28


,tipo_de_contrato,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Prestación de servicios,388023,105320,27.14,27.00,27.28
1,Otro,16849,6387,37.91,37.18,38.64
2,Decreto 092 de 2017,15978,4050,25.35,24.68,26.03
3,Suministros,10780,2292,21.26,20.50,22.04
4,Compraventa,8917,1959,21.97,21.12,22.84
5,Obra,4081,2116,51.85,50.32,53.38
6,Arrendamiento de inmuebles,3494,1178,33.71,32.17,35.30
7,Interventoría,1043,749,71.81,69.01,74.46
8,Comodato,1040,171,16.44,14.31,18.82
9,Consultoría,889,377,42.41,39.20,45.68


,len_objeto,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,<=50,5863,3398,57.96,56.69,59.21
1,51-100,17915,6154,34.35,33.66,35.05
2,101-200,142903,36088,25.25,25.03,25.48
3,201-400,239544,67642,28.24,28.06,28.42
4,>400,46269,11766,25.43,25.03,25.83


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\517279465.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultado: confirmada** para H7 (`docs/01_definicion_problema.md` §3). Los segmentos y familias UNSPSC, el tipo de contrato y la longitud del objeto separan tasas: segmento 85 40,3 % frente a 82 17,7 %; interventoría 71,9 % frente a comodato 18,6 %; `len_objeto` <=50 58,5 %.

**Decisión:** `tipo_de_contrato`, `unspsc_segmento`, `unspsc_familia` y `len_objeto` entran en `FEATURES` (`features.FEATURES_CAT`, `features.FEATURES_NUM`); las dos UNSPSC se derivan del código de categoría (`features.construir_features`). Las tres categóricas se codifican con `OrdinalEncoder(max_categories=200)` (`preprocesamiento.construir_preprocesador`); `len_objeto` es numérica cruda (`features.construir_features`) y no se usa el texto crudo del objeto. `len_objeto` se conserva con seguimiento en la evaluación con entidades no vistas: es un indicador indirecto de la entidad (cargos de hospitales), no una fuga (`docs/EDA_INSIGHTS_PREPROCESAMIENTO.md`).

**Alternativas:**
- Excluir `len_objeto` y dejar solo tipo/UNSPSC.
- Usar el texto crudo del objeto.
- Agrupar todos los tipos raros en “Otro” a mano (el `OrdinalEncoder` ya agrupa las categorías infrecuentes en un código).
- Categorías nativas de HGB: descartadas por la explicación SHAP (`resultados/j3_decisiones.md`).

**Por qué:** Tipo y UNSPSC separan tasas muy distintas; `len_objeto` corto se asocia con objetos de personal de hospitales y puede estar capturando la entidad. Es señal disponible al firmar, pero su peso debe revisarse fuera de las entidades vistas.

**Evidencia principal (train 2019–2023):** Tabla 6.7.1.b: segmento 85 40,3 % (n=13.544), 72 38,6 % (n=4.408), 93 34,3 %, 80 28,5 %, 82 17,7 %. Tabla 6.7.2.b: familia 8510 45,8 % (n=9.672), 8110 41,0 %, 9315 38,5 %, 9314 32,7 %, 8011 28,8 %, 8111 21,6 %. Tabla 6.7.3.b: interventoría 71,9 % (n=723), obra 53,2 % (n=2.409), consultoría 45,3 %, prestación de servicios 28,2 % (n=224.529), compraventa 20,5 %, suministros 20,3 %, comodato 18,6 %. Tabla 6.7.4.b: <=50 caracteres 58,5 % (n=4.385), 51–100 35,5 %, 101–200 25,7 %, 201–400 29,6 %, >400 26,8 %.

**Contraste 2019–2025:** Tabla 6.7.1: 72 38,9 %, 85 38,5 %, 80 27,0 %, Otras 22,2 %; **cambia el orden** entre 85 y 72. Tabla 6.7.2: 8510 43,7 %, 8110 38,9 %, 9314 32,9 %, 9315 32,1 %; **cambian de orden 9315 y 9314 y entra 8111 en lugar de 8512**. Tabla 6.7.3: interventoría 71,8 %, obra 51,9 %, prestación de servicios 27,1 %, comodato 16,4 %; **Decreto 092 pasa de -2,3 pp a +6,6 pp** y sube en el orden. Tabla 6.7.4: <=50 58,0 % frente a 201–400 28,2 %; la relación sigue sin ser monótona. La dirección general se mantiene en las cuatro.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §6.7

## 7. Privacidad del proveedor

**Qué se observa.** El 84,25 % de los contratos tiene proveedor con cédula de ciudadanía, 15,21 % con NIT, 0,16 % "No Definido" y 0,19 % "Otro". La composición cambia fuerte en 2019 (75,55 % cédula / 23,61 % NIT) y se estabiliza desde 2023 (84,56–85,34 % / 14,14–14,98 %). Es decir, más de cuatro de cada cinco contratos tienen un proveedor con cédula, un dato personal.

**Decisión.**
- **Decisión:** excluir la identidad y los datos del proveedor (documento, nombre, `codigo_proveedor`, NIT adjudicado); conservar solo `tipodocproveedor`.
- **Alternativas:** historial de prórrogas del proveedor; identificador cifrado (hash); usar solo personas jurídicas (NIT); excluir.
- **Por qué:** la cédula es dato personal (Ley 1581 de 2012); el hash sigue siendo un dato personal seudonimizado y un puntaje por proveedor se convierte en un ranking de personas. El tipo de documento no identifica.
- **Evidencia:** distribución de `tipodocproveedor` y su evolución por año.
- **Costo:** se pierde una posible señal (historial del proveedor) cuyo tamaño no se midió [NO EVALUADO].


In [23]:
tipo_doc = eda.resumen_tipodocproveedor(base)
display(tipo_doc.round(2))
eda.graficar_tipodocproveedor(tipo_doc, titulo="Tipo de documento del proveedor (base v1)")
plt.show()
eda.graficar_evolucion_categorias(base, ["tipodocproveedor"], top=4, ncols=1,
                                  titulo="Evolución del tipo de documento del proveedor")
plt.show()


,tipo,n,pct,pct_acumulado
0,Cédula de Ciudadanía,381242,84.25,84.25
1,NIT,68844,15.21,99.47
2,Otro,878,0.19,99.66
3,No Definido,730,0.16,99.82
4,Cédula de Extranjería,607,0.13,99.96
5,Permiso por Protección Temporal,84,0.02,99.98
6,Pasaporte,45,0.01,99.99
7,Tarjeta de Identidad,42,0.01,100.00
8,Permiso especial de permanencia,17,0.00,100.00
9,Registro Civil,5,0.00,100.00


C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1817356180.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\kevineafit\AppData\Local\Temp\ipykernel_75160\1817356180.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2 Tasa de y por tipo de documento del proveedor

Se muestra la tasa de `y` por `tipodocproveedor` (Wilson) y su evolución por año. Solo el tipo de documento, nunca la identidad del proveedor.


In [24]:
# §7.2: solo el tipo de documento; nunca la identidad del proveedor.
tabla_doc = eda.tasas_en_pct(eda.tasa_por_grupo(base, "tipodocproveedor")).round(2)
tabla_doc = tabla_doc.sort_values("n", ascending=False).reset_index(drop=True)
display(tabla_doc)

evol_doc = (
    base.pivot_table(
        index="tipodocproveedor",
        columns="anio_firma",
        values="y",
        aggfunc="mean",
    ) * 100
).round(2)
evol_doc = evol_doc.loc[tabla_doc["tipodocproveedor"]]
display(evol_doc)


,tipodocproveedor,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Cédula de Ciudadanía,381242,104261,27.35,27.21,27.49
1,NIT,68844,19862,28.85,28.51,29.19
2,Otro,878,270,30.75,27.79,33.88
3,No Definido,730,442,60.55,56.96,64.03
4,Cédula de Extranjería,607,168,27.68,24.27,31.37
5,Permiso por Protección Temporal,84,28,33.33,24.18,43.95
6,Pasaporte,45,6,13.33,6.26,26.18
7,Tarjeta de Identidad,42,6,14.29,6.72,27.84
8,Permiso especial de permanencia,17,4,23.53,9.55,47.26
9,Registro Civil,5,1,20.00,3.62,62.45


anio_firma,2019,2020,2021,2022,2023,2024,2025
tipodocproveedor,,,,,,,
Cédula de Ciudadanía,22.85,28.63,27.21,29.31,30.37,23.90,26.67
NIT,18.84,32.23,30.47,29.77,27.52,28.58,29.41
Otro,37.50,35.90,26.67,28.17,29.19,29.73,35.54
No Definido,57.69,60.67,67.24,63.64,54.64,64.89,56.05
Cédula de Extranjería,62.50,24.00,21.95,28.57,29.91,22.58,30.97
Permiso por Protección Temporal,NaN,NaN,0.00,100.00,29.41,40.74,27.03
Pasaporte,0.00,25.00,0.00,7.14,20.00,33.33,50.00
Tarjeta de Identidad,NaN,0.00,28.57,8.33,16.67,0.00,22.22
Permiso especial de permanencia,NaN,NaN,100.00,0.00,16.67,50.00,0.00


**Decisión:** Conservar solo `tipodocproveedor` como feature categórica (`features.FEATURES_CAT`), codificada con `OrdinalEncoder(max_categories=200)` en la familia `arbol` (`preprocesamiento.construir_preprocesador`); excluir la identidad y los datos del proveedor (documento, nombre, código, NIT adjudicado).

**Alternativas:**
- Incluir el historial de prórrogas del proveedor.
- Usar un identificador cifrado o hash.
- Usar el proveedor solo para personas jurídicas (NIT).

**Por qué:** La cédula de ciudadanía domina la muestra y es un dato personal; un puntaje por proveedor se convertiría en un ranking de personas. El tipo de documento no identifica al proveedor y las diferencias entre cédula y NIT son pequeñas (28,8 % vs 28,7 % en train).

**Evidencia principal (train 2019–2023):** Tabla 7.2.1.b (n = 261.831; y = 28,8 %): cédula de ciudadanía `y` 28,8 % (n=219.724), NIT `y` 28,7 % (n=40.593), Otro `y` 29,6 % (n=564), Otras `y` 43,7 % (n=950).

**Contraste 2019–2025:** Tabla 7.2.1: cédula de ciudadanía 84,3 % (n=381.242) con `y` 27,3 %; NIT 15,2 % (n=68.844) con 28,9 %; No Definido 0,2 % (n=730) con 60,5 %. **Cambia la comparación**: en train NIT y cédula quedan casi iguales (28,7 % vs 28,8 %), mientras que en 2019–2025 el NIT está 1,6 pp por encima; `Otras` mezcla No Definido y Cédula de Extranjería, así que su 43,7 % no es comparable. Tabla 7.2.2: las tasas anuales de las categorías pequeñas son inestables; “Otras” pasa de 0,0 % (n=2) en 2019 a 26,9 % (n=52) en 2025.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §7.2

## 8. Fuga de información

La lista de columnas de fuga y su estado está en `docs/LISTA_FUGA.md`. La medición con modelo (ROC-AUC con y sin fuga) la hace J1 con la v1; aquí solo se deja una comprobación descriptiva de `estado_contrato` contra `y`, sin usar la columna como feature.


In [25]:
# §8: lectura de solo lectura de estado_contrato; no se escribe nada.
RUTA_RAW_CONTRATOS = Path(
    r"C:\Users\kevineafit\OneDrive\Escritorio\SEM2026-2\ML\workshops\repositorio\fundamentos-aa-2026\final-project\data\raw\f2\contratos"
)
archivos_contratos = sorted(RUTA_RAW_CONTRATOS.rglob("*.parquet"))
estado = pd.concat(
    [pd.read_parquet(f, columns=["id_contrato", "estado_contrato"]) for f in archivos_contratos],
    ignore_index=True,
).drop_duplicates(subset="id_contrato")
base_estado = base[["id_contrato", "y"]].merge(
    estado, on="id_contrato", how="left", validate="one_to_one"
)
tabla_estado = eda.tasas_en_pct(
    eda.tasa_por_grupo(base_estado, "estado_contrato")
).round(2)
display(tabla_estado.sort_values("n", ascending=False).reset_index(drop=True))
print("Contratos con estado_contrato:", int(base_estado["estado_contrato"].notna().sum()), "de", len(base_estado))


,estado_contrato,n,positivos,tasa_pct,ic_inf_pct,ic_sup_pct
0,Cerrado,166839,42613,25.54,25.33,25.75
1,En ejecución,118069,353,0.30,0.27,0.33
2,Modificado,81391,62574,76.88,76.59,77.17
3,terminado,73006,18988,26.01,25.69,26.33
4,Aprobado,10997,21,0.19,0.12,0.29
5,cedido,1802,267,14.82,13.25,16.53
6,Suspendido,383,232,60.57,55.60,65.34
7,enviado Proveedor,5,0,0.00,-0.00,43.45
8,En aprobación,2,0,0.00,0.00,65.76


Contratos con estado_contrato: 452494 de 452494


**Decisión:** Excluir de `FEATURES` las columnas de fuga confirmada y los agregados de Modificaciones (`features.COLUMNAS_FUGA`); `estado_contrato` se usa solo para EDA, no como feature ni como respaldo. `el_contrato_puede_ser_prorrogado` sigue fuera de `FEATURES` por la decisión del PLAN (`features.FEATURES_EXCLUIDAS`); la ablación de J3 la respalda (agregarla mueve el PR-AUC 2024 en −0,0010, dentro del ruido; `resultados/j3_decisiones.md`).

**Alternativas:**
- Incluir `estado_contrato` con regularización.
- Usar las columnas solo para evaluación o para definir la etiqueta.
- Incluir los agregados de Modificaciones como features.

**Por qué:** Esas columnas contienen hechos posteriores a la firma; su AUC alto es consistente con fuga, no con señal utilizable al firmar. El AUC baja de 0,943 con `y_dias` a 0,820 con `y` porque cambió la etiqueta, no porque deje de ser fuga: “Modificado” tiene 76,9 % de positivos frente a 0,3 % de “En ejecución”, es decir, el estado se actualiza con el desenlace.

**Evidencia:** Tabla 8.1 (2019–2025): “Modificado” 76,9 % de `y` (n=81.391) frente a “En ejecución” 0,3 % (n=118.069) y “Aprobado” 0,2 % (n=10.997). Tabla 8.2: `estado_contrato` AUC 0,820 vs `y` y 0,943 vs `y_dias` (legacy 0,94); `razon_dur` 0,952; `mod_n` 0,9996; `dias_adicionados` 1,0000 vs `y_dias`.

**Cadena de fuga del modelo:** con el HGB limpio, ROC-AUC 0,781 → 0,917 → 0,984 y PR-AUC 0,584 → 0,851 → 0,967: el primer salto agrega los 6 montos del crudo de Contratos y el segundo la fuga evidente (estado, valor y fecha de fin). Cifras en `resultados/j3_extra_fuga_montos.csv` (filas `final_limpio`, `final_con_6_montos`, `final_con_fuga_evidente`) y en `resultados/j3_decisiones.md`.

**Cautela:** Los AUC de la Tabla 8.2 son descriptivos, no una métrica de validación; no hay train/test ni modelo en esa sección. La cadena de fuga es un contraste sobre la base v1_A con el crudo de Contratos unido por `id_contrato` solo para medir; esas columnas no entran en la base ni en `FEATURES`.

Detalle y trazabilidad al código: docs/eda_interpretaciones.md §8

## 9. Resumen de decisiones

| ID | Decisión | Alternativas | Por qué | Evidencia | Va a |
|---|---|---|---|---|---|
| D2.1 | Imputar en el Pipeline con mediana/moda + indicadores de bloque | Borrar filas; imputar con 0 | Faltantes bajos y en bloque; borrar sesga | 349 sin proceso, 4.059 sin inicio, 799 sin duración, 3.806 sin UNSPSC | J2 |
| D2.2 | Duración ≤0 → faltante y recorte [1, 3.650]; precio negativo → flag (ya queda en 0 en `log_precio_base`) | Dejar crudo; borrar outliers | 5,43 % de duraciones no positivas y colas de 4.078,9 M | 24.566 ≤0, 125 >3.650, 10 precios negativos | J2 |
| D2.3 | Agrupar raras <1 % en `OTROS`; HGB nativo/tope; `nit_entidad` con TargetEncoder | One-hot completo; dejar texto | Cardinalidad alta y 76,59 % de filas en entidades con <1 % | 3.034 entidades, 735 ciudades | J2 |
| D2.4 | Eliminar constantes/redundantes | Mantener las 18 numéricas | Duplican la misma señal | 1 constante (`p_proveedores_que_manifestaron`), 99,66 %/99,48 % iguales, r 0,994/0,992 | J2 |
| D3.1 | `log1p` en conteos + `RobustScaler` para kNN/SVM; crudo para HGB | Estandarizar sin log | Zero-inflación y colas largas | 90,4–99,8 % ceros en competencia | J2 |
| D3.2 | No usar el año como feature; ajustar solo con train | Usar año; mezclar train/test | Deriva fuerte 2019→2025 | Composición y competencia cambian | J2/J3 |
| D3.3 | Quitar redundantes para kNN/SVM; mantener para HGB | PCA | Colinealidad duplica peso | r 0,994 y 0,992 | J2/J3 |
| D3.4 | Calendario: flag de trimestre + mes cíclico o uno solo para distancia | Mes ordinal | `mes_firma` y el flag correlacionan 0,695–0,722 | r = 0,695/0,722 | J2 |
| D4.1 | Entrenar/evaluar solo maduros; `eval_*` nunca como feature | Usar todos | 99,2 % maduro; 3.455 inmaduros | Madurez 97,64–99,99 % por año | J2/J3 |
| D7.1 | Excluir identidad del proveedor; conservar solo el tipo de documento | Hash; historial; solo NIT | 84,25 % de los contratos tiene proveedor con cédula | Distribución `tipodocproveedor` | J3/app |

Las decisiones de §4.3, §5, §6 (H1–H7), §7.2 y §8 están en cada sección y en `docs/eda_interpretaciones.md`. El detalle para Jose está en `docs/EDA_INSIGHTS_PREPROCESAMIENTO.md`.

In [26]:
print("Secciones con tablas y gráficas: 2, 3, 4, 5, 6, 7 y 8 (fuga descriptiva).")
print("Interpretación cerrada en 4.3, 5, 6, 7.2 y 8; la medición con modelo de §8 la hace J1 con la v1.")


Secciones con tablas y gráficas: 2, 3, 4, 5, 6, 7 y 8 (fuga descriptiva).
Interpretación cerrada en 4.3, 5, 6, 7.2 y 8; la medición con modelo de §8 la hace J1 con la v1.
